In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-11-04 02:26:13--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-11-04 02:26:13 (32.5 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [2]:
# lê o arquivo
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [4]:
# mostra os primeiros 1000 caracteres
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
# todos os caracteres únicos que aparecem no texto
chars = sorted(list(set(text)))  # lista ordenada sem repetição
vocab_size = len(chars)  # 65 caracteres = tamanho do vocabulário
print(''.join(chars))  # mostra o vocabulário
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
# mapeia caracteres para inteiros e vice-versa
stoi = { ch:i for i,ch in enumerate(chars) }  # caractere para índice
itos = { i:ch for i,ch in enumerate(chars) }  # índice para caractere
encode = lambda s: [stoi[c] for c in s] # recebe uma string e devolve uma lista de inteiros
decode = lambda l: ''.join([itos[i] for i in l]) # recebe uma lista de inteiros e devolve uma string

print(encode("hii there"))  # texto para números
print(decode(encode("hii there")))  # volta ao texto original

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [7]:
# codifica o texto inteiro e guarda
import torch

data = torch.tensor(encode(text), dtype=torch.long)  # um inteiro por caractere
print(data.shape, data.dtype)
print(data[:1000]) # os 1000 caracteres vistos antes, como o GPT os enxerga

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [8]:
# divide os dados em treino e validação
n = int(0.9*len(data)) # 90% para treino
train_data = data[:n]
val_data = data[n:]  # 10% para validação

In [9]:
block_size = 8  # tamanho do contexto
train_data[:block_size+1]  # 9 caracteres: 8 entradas + 1 alvo

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x = train_data[:block_size]  # entradas
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]  # contexto até a posição t
    target = y[t]  # próximo caractere a prever
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [11]:
torch.manual_seed(1337)  # semente
batch_size = 4
block_size = 8

def get_batch(split):  # sorteia um lote de treino ou validação
    # gera um lote pequeno de entradas x e alvos y
    data = train_data if split == 'train' else val_data  # escolhe treino ou validação
    ix = torch.randint(len(data) - block_size, (batch_size,))  # posições iniciais aleatórias
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')  # pega um lote de treino
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # dimensão do lote
    for t in range(block_size): # dimensão do tempo
        context = xb[b, :t+1]  # contexto até a posição t
        target = yb[b,t]  # caractere alvo
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [12]:
print(xb) # entrada para o transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)  # semente

class BigramLanguageModel(nn.Module):  # prevê o próximo caractere só pelo atual

    def __init__(self, vocab_size):
        super().__init__()
        # cada token lê direto de uma tabela os logits do próximo token
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)  # tabela

    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)  # achata
            targets = targets.view(B*T)  # achata os alvos
            loss = F.cross_entropy(logits, targets)  # perda de entropia cruzada

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens): # obtém as previsões
            logits, loss = self(idx)  # forward com o contexto atual
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1) # aplica softmax para obter probabilidades
            idx_next = torch.multinomial(probs, num_samples=1) # amostra da distribuição
            idx = torch.cat((idx, idx_next), dim=1) # adiciona o índice amostrado à sequência
        return idx

m = BigramLanguageModel(vocab_size)  # cria o modelo
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))  # gera 100 caracteres (aleatórios: modelo não treinado)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [14]:
# cria um otimizador
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [15]:
batch_size = 32  # lote maior para o treino
for steps in range(100):

    xb, yb = get_batch('train')  # lote de treino

    # calcula a perda
    logits, loss = m(xb, yb)  # forward
    optimizer.zero_grad(set_to_none=True)  # zera os gradientes
    loss.backward()  # retropropagação
    optimizer.step()  # atualiza os pesos

print(loss.item())

4.65630578994751


In [16]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))  # gera 500 caracteres


oTo.JUZ!!zqe!
xBP qbs$Gy'AcOmrLwwt
p$x;Seh-onQbfM?OjKbn'NwUAW -Np3fkz$FVwAUEa-wzWC -wQo-R!v -Mj?,SPiTyZ;o-opr$mOiPJEYD-CfigkzD3p3?zvS;ADz;.y?o,ivCuC'zqHxcVT cHA
rT'Fd,SBMZyOslg!NXeF$sBe,juUzLq?w-wzP-h
ERjjxlgJzPbHxf$ q,q,KCDCU fqBOQT
SV&CW:xSVwZv'DG'NSPypDhKStKzC -$hslxIVzoivnp ,ethA:NCCGoi
tN!ljjP3fwJMwNelgUzzPGJlgihJ!d?q.d
pSPYgCuCJrIFtb
jQXg
pA.P LP,SPJi
DBcuBM:CixjJ$Jzkq,OLf3KLQLMGph$O 3DfiPHnXKuHMlyjxEiyZib3FaHV-oJa!zoc'XSP :CKGUhd?lgCOF$;;DTHZMlvvcmZAm;:iv'MMgO&Ywbc;BLCUd&vZINLIzkuTGZa
D.?


In [17]:
torch.manual_seed(42)  # semente
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)  # cada linha soma 1
b = torch.randint(0,10,(3,2)).float()  # valores aleatórios 3x2
c = a @ b  # cada linha de c é a média das linhas de b até ela
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [18]:
torch.manual_seed(1337)  # semente
B,T,C = 4,8,2 # lote, tempo, canais
x = torch.randn(B,T,C)  # tensor aleatório (4, 8, 2)
x.shape  # confere o formato

torch.Size([4, 8, 2])

In [19]:
xbow = torch.zeros((B,T,C))  # média dos tokens anteriores
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev, 0)  # média no tempo


In [20]:
wei = torch.tril(torch.ones(T, T))  # triangular inferior
wei = wei / wei.sum(1, keepdim=True)  # cada linha soma 1
xbow2 = wei @ x
torch.allclose(xbow, xbow2)

False

In [21]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))  # afinidades iniciais todas zero
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)  # apllicando o softmax
xbow3 = wei @ x
torch.allclose(xbow, xbow3)  # mesmo resultado de novo


False

In [22]:
torch.manual_seed(1337)  # semente
B,T,C = 4,8,32 # lote, tempo, canais
x = torch.randn(B,T,C)  # entrada aleatória

head_size = 16  # dimensão da cabeça
key = nn.Linear(C, head_size, bias=False)  # o que cada token contém
query = nn.Linear(C, head_size, bias=False)  # o que cada token procura
value = nn.Linear(C, head_size, bias=False)  # o que cada token comunica
k = key(x)
q = query(x)
wei =  q @ k.transpose(-2, -1)

tril = torch.tril(torch.ones(T, T))  # máscara causal
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))  # impede olhar o futuro
wei = F.softmax(wei, dim=-1)  # normaliza as afinidades

v = value(x)
out = wei @ v  # agregação ponderada dos valores
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [23]:
wei[0]  # pesos do primeiro exemplo

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [24]:
k = torch.randn(B,T,head_size)  # chaves aleatórias com variância 1
q = torch.randn(B,T,head_size)  # queries aleatórias com variância 1
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [25]:
k.var()  # variancia

tensor(1.0449)

In [26]:
q.var() # variancia

tensor(1.0700)

In [27]:
wei.var() # variancia

tensor(1.0918)

In [28]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [29]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1)

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [30]:
class LayerNorm1d:

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps  # evita divisão por zero
    self.gamma = torch.ones(dim)  # escala
    self.beta = torch.zeros(dim)  # deslocamento

  def __call__(self, x):
    # passagem direta (forward)
    xmean = x.mean(1, keepdim=True) # média de cada exemplo
    xvar = x.var(1, keepdim=True) # variância de cada exemplo
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normaliza para variância unitária
    self.out = self.gamma * xhat + self.beta  # aplica escala e deslocamento
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]  # parâmetros treináveis

torch.manual_seed(1337)
module = LayerNorm1d(100)  # camada para vetores de 100 dimensões
x = torch.randn(32, 100) # lote de 32 vetores de 100 dimensões
x = module(x)  # normaliza cada linha
x.shape

torch.Size([32, 100])

In [31]:
x[:,0].mean(), x[:,0].std() # média e desvio de uma feature ao longo do lote

(tensor(0.1469), tensor(0.8803))

In [32]:
x[0,:].mean(), x[0,:].std() # média e desvio das features de um exemplo

(tensor(-9.5367e-09), tensor(1.0000))

In [33]:
import torch
import torch.nn as nn
from torch.nn import functional as F

batch_size = 16
block_size = 32
max_iters = 5000  # total de passos de treino
eval_interval = 100  # avalia a cada 100 passos
learning_rate = 1e-3  # taxa de aprendizado
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200  # lotes usados em cada avaliação
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0

torch.manual_seed(1337)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))  # lista ordenada sem repetição
vocab_size = len(chars)  # tamanho do vocabulário

# mapeia caracteres para inteiros e vice-versa
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # codificador
decode = lambda l: ''.join([itos[i] for i in l]) # decodificador

# divisão treino/validação
data = torch.tensor(encode(text), dtype=torch.long)  # um inteiro por caractere
n = int(0.9*len(data)) # 90% para treino
train_data = data[:n]
val_data = data[n:]  # 10% para validação

# carregamento dos dados
def get_batch(split):
    # gera um lote pequeno de entradas x e alvos y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))  # posições iniciais aleatórias
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])  # alvos deslocados em 1
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():  # perda média em treino e validação
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)  # guarda as perdas de cada lote
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()  # média das perdas
    model.train()  # volta ao modo treino
    return out

class Head(nn.Module):
    """ uma cabeça de self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)  # dropout nos pesos de atenção

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        # calcula os scores de atenção
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)  # dropout na atenção
        # faz a agregação ponderada dos valores
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ várias cabeças de self-attention em paralelo """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)  # projeção de saída
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # concatena as saídas das cabeças
        out = self.dropout(self.proj(out))  # projeta e aplica dropout
        return out

class FeedFoward(nn.Module):
    """ uma camada linear seguida de uma não linearidade """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),  # expande 4x
            nn.ReLU(),  # não linearidade
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ bloco transformer: comunicação seguida de computação """

    def __init__(self, n_embd, n_head): # dimensão do embedding, número de cabeças
        super().__init__()
        head_size = n_embd // n_head  # 64 // 4 = 16 por cabeça
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # cada token lê direto de uma tabela os logits do próximo token
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)  # embedding dos tokens
        self.position_embedding_table = nn.Embedding(block_size, n_embd)  # embedding das posições
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # layer norm final
        self.lm_head = nn.Linear(n_embd, vocab_size)  # projeta para o vocabulário

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)  # perda de entropia cruzada

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # vira
            probs = F.softmax(logits, dim=-1)  # aplica softmax para obter probabilidades
            idx_next = torch.multinomial(probs, num_samples=1) # amostra da distribuição
            idx = torch.cat((idx, idx_next), dim=1) # adiciona o índice amostrado à sequência
        return idx

model = BigramLanguageModel()  # instancia o modelo
m = model.to(device)

print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters') # mostra o número de parâmetros do modelo

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)  # cria um otimizador do PyTorch

for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()  # avalia treino e validação
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # amostra um lote de dados
    xb, yb = get_batch('train')  # lote de treino

    # calcula a perda
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)  # zera os gradientes
    loss.backward()
    optimizer.step()

# gera texto com o modelo
context = torch.zeros((1, 1), dtype=torch.long, device=device)  # começa pelo token 0
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))  # gera 2000 caracteres


0.209729 M parameters
step 0: train loss 4.4116, val loss 4.4022
step 100: train loss 2.6568, val loss 2.6670
step 200: train loss 2.5090, val loss 2.5058
step 300: train loss 2.4194, val loss 2.4334
step 400: train loss 2.3501, val loss 2.3568
step 500: train loss 2.2963, val loss 2.3129
step 600: train loss 2.2410, val loss 2.2501
step 700: train loss 2.2057, val loss 2.2191
step 800: train loss 2.1633, val loss 2.1860
step 900: train loss 2.1242, val loss 2.1498
step 1000: train loss 2.1027, val loss 2.1298
step 1100: train loss 2.0692, val loss 2.1183
step 1200: train loss 2.0386, val loss 2.0797
step 1300: train loss 2.0276, val loss 2.0652
step 1400: train loss 1.9925, val loss 2.0370
step 1500: train loss 1.9702, val loss 2.0302
step 1600: train loss 1.9645, val loss 2.0487
step 1700: train loss 1.9421, val loss 2.0143
step 1800: train loss 1.9091, val loss 1.9953
step 1900: train loss 1.9085, val loss 1.9874
step 2000: train loss 1.8861, val loss 1.9957
step 2100: train loss 1.